![Banner CEAP](https://www.camara.leg.br/internet/deputado/img/logo-camara.png)

# 💳 Fast Track — 05. Bronze: CEAP Despesas

**Ingestão de Dados Brutos — API Câmara dos Deputados**

| Item | Descrição |
|---|---|
| **Objetivo** | Ingerir despesas da Cota Parlamentar (CEAP) |
| **Entidade** | Despesas CEAP (passagens, combustível, telefonia, etc) |
| **Endpoint API** | `https://dadosabertos.camara.leg.br/api/v2/deputados/{id}/despesas` |
| **Tabela Destino** | `uc_fast_track.bronze.ceap_despesas_raw` |
| **Modo** | APPEND |
| **Checkpoint** | Por deputado + mês/ano |
| **Formato** | Payload JSON raw + colunas técnicas |
| **Responsável** | Ernesto Bassoli |

---

## 📋 O Que é CEAP?

**Cota para Exercício da Atividade Parlamentar (CEAP)** é um valor mensal que cada deputado recebe para custear despesas relacionadas ao exercício do mandato.

**Tipos de despesa:**
* Passagens aéreas
* Combustível e lubrificantes
* Telefonia, correio e internet
* Divulgação da atividade parlamentar
* Hospedagem e alimentação
* Serviços de segurança
* Consultorias e assessorias

**Importância:** Transparência na utilização de recursos públicos pelos parlamentares.

---

## ⚠️ IMPORTANTE: Endpoint por Deputado

Diferente dos outros endpoints, CEAP **NÃO tem endpoint agregado**. É necessário:
1. Buscar lista de deputados
2. Para cada deputado, chamar `/deputados/{id}/despesas`
3. Filtrar por período (ano/mês)

Isso requer uma lógica diferente de checkpoint e iteração.

## ⚙️ Bloco 1: Configuração

In [0]:
print("🔗 Recuperando variáveis...")

try:
    print(f"✅ Load ID: {load_id}")
    print(f"✅ Ambiente: {env}")
    print(f"✅ Catálogo: {catalog}")
except NameError as e:
    raise Exception(f"Execute notebook 00 primeiro: {e}")

entity_name = "ceap_despesas"
api_base_url = "https://dadosabertos.camara.leg.br/api/v2"
table_bronze = f"{catalog}.{schema_bronze}.{entity_name}_raw"
page_size = 100
max_retries = 3

print(f"\n✅ Entidade: {entity_name}")
print(f"✅ Tabela: {table_bronze}")
print("="*70)

## 📦 Bloco 2: Importações

In [0]:
import requests, json, time, uuid, hashlib
from datetime import datetime
from pyspark.sql.types import *
print("✅ Bibliotecas OK")

## 🔌 Bloco 3: Função Extração

Esta função itera sobre deputados e busca despesas de cada um.

In [0]:
def get_ceap_from_api(deputados_ids, ano=None, mes=None):
    """
    Busca despesas CEAP de uma lista de deputados.
    
    Args:
        deputados_ids: Lista de IDs de deputados
        ano: Ano para filtrar (ex: 2026)
        mes: Mês para filtrar (ex: 4)
    """
    print(f"🔌 Extraindo CEAP de {len(deputados_ids)} deputados")
    if ano:
        print(f"   Período: {mes}/{ano}" if mes else f"   Ano: {ano}")
    
    all_despesas = []
    total_reqs = 0
    
    for idx, dep_id in enumerate(deputados_ids, 1):
        print(f"\n👤 [{idx}/{len(deputados_ids)}] Deputado ID {dep_id}")
        endpoint = f"{api_base_url}/deputados/{dep_id}/despesas"
        
        pagina = 1
        has_more = True
        
        while has_more:
            params = {"pagina": pagina, "itens": page_size, "ordem": "ASC"}
            if ano:
                params["ano"] = ano
            if mes:
                params["mes"] = mes
            
            req_id = str(uuid.uuid4())
            start = time.time()
            success = False
            resp_data = None
            status = None
            err = None
            
            for attempt in range(1, max_retries + 1):
                try:
                    r = requests.get(endpoint, params=params, timeout=30)
                    status = r.status_code
                    if r.status_code == 200:
                        resp_data = r.json()
                        success = True
                        break
                    else:
                        err = f"HTTP {r.status_code}"
                        if attempt < max_retries:
                            time.sleep(3 * attempt)
                except Exception as e:
                    err = str(e)
                    if attempt < max_retries:
                        time.sleep(3 * attempt)
            
            resp_time = int((time.time() - start) * 1000)
            total_reqs += 1
            
            # Log request
            spark.createDataFrame([{
                "request_id": req_id, "load_id": load_id, "entity_name": entity_name,
                "endpoint": f"{endpoint}?pagina={pagina}", "http_method": "GET",
                "http_status": status, "response_time_ms": resp_time, "success": success,
                "error_message": err, "request_timestamp": datetime.now()
            }]).write.mode("append").saveAsTable(f"{catalog}.{schema_ops}.ingestion_requests")
            
            if not success:
                print(f"   ❌ Erro após {max_retries} tentativas: {err}")
                break
            
            data_page = resp_data.get("dados", [])
            print(f"   Pág {pagina}: {len(data_page)} despesas ({resp_time}ms)")
            
            # Enriquece cada despesa com deputado_id
            for desp in data_page:
                desp["_deputado_id"] = dep_id
            
            all_despesas.extend(data_page)
            
            links = resp_data.get("links", [])
            has_more = any(l.get("rel") == "next" for l in links) and len(data_page) > 0
            if has_more:
                pagina += 1
                time.sleep(0.3)
            else:
                break
        
        # Pausa entre deputados
        if idx < len(deputados_ids):
            time.sleep(0.5)
    
    print(f"\n✅ Extração completa: {total_reqs} requests, {len(all_despesas)} despesas")
    return all_despesas

print("✅ Função definida")

## 👥 Bloco 4: Buscar Lista de Deputados

Precisamos dos IDs dos deputados para buscar suas despesas.

In [0]:
print("👥 Buscando lista de deputados...\n")

# Busca deputados da tabela Bronze (se já ingerida)
try:
    df_deps = spark.sql(f"""
        SELECT DISTINCT CAST(record_id AS INT) as deputado_id
        FROM {catalog}.{schema_bronze}.deputados_raw
        WHERE record_id IS NOT NULL AND record_id != 'unknown'
        ORDER BY deputado_id
    """)
    
    deputados_ids = [row["deputado_id"] for row in df_deps.collect()]
    print(f"✅ {len(deputados_ids)} deputados encontrados na tabela Bronze")
    
    if len(deputados_ids) == 0:
        print("⚠️ Nenhum deputado encontrado - execute notebook 01_bronze_deputados primeiro")
        dbutils.notebook.exit("NO_DEPUTADOS")
    
    # Para testes/desenvolvimento, limitar a 10 deputados
    # Remover esta linha em produção para processar todos
    if env == "dev":
        deputados_ids = deputados_ids[:10]
        print(f"⚠️ Modo DEV: limitando a {len(deputados_ids)} deputados")
    
    print(f"\n🎯 Processará {len(deputados_ids)} deputados")
    
except Exception as e:
    print(f"❌ Erro ao buscar deputados: {e}")
    print("   Certifique-se que o notebook 01_bronze_deputados foi executado")
    raise

print("="*70)

## 🔖 Bloco 5: Checkpoint

Para CEAP, checkpoint é diferente: controlamos por período (ano/mês) já processado.

In [0]:
print("🔖 Definindo período...\n")

# Para este exemplo, vamos buscar despesas do mês/ano do run_date
import re
run_date_match = re.match(r'(\d{4})-(\d{2})', run_date)
if run_date_match:
    ano_processo = int(run_date_match.group(1))
    mes_processo = int(run_date_match.group(2))
else:
    # Fallback: usar ano/mês atual
    from datetime import datetime as dt
    hoje = dt.now()
    ano_processo = hoje.year
    mes_processo = hoje.month

print(f"📌 Período: {mes_processo:02d}/{ano_processo}")
print(f"   Buscará despesas deste período para todos os deputados")
print("="*70)

## 🗄️ Bloco 6: Criar Tabela Bronze

In [0]:
print(f"🗄️ Criando {table_bronze}...\n")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_bronze} (
    payload STRING COMMENT 'Payload JSON raw',
    record_id STRING COMMENT 'ID da despesa',
    deputado_id STRING COMMENT 'ID do deputado',
    record_hash STRING COMMENT 'Hash SHA256',
    load_id STRING COMMENT 'UUID execução',
    ingestion_timestamp TIMESTAMP,
    ingestion_date DATE,
    source_endpoint STRING,
    env STRING
)
USING DELTA
PARTITIONED BY (ingestion_date)
COMMENT 'Bronze - CEAP Despesas Raw'
""")

print("✅ Tabela OK")
print("="*70)

## 📥 Bloco 7: Ingestão

In [0]:
print("📥 Iniciando ingestão...\n")
ingest_start = time.time()

data_list = get_ceap_from_api(deputados_ids, ano=ano_processo, mes=mes_processo)
num_records = len(data_list)

print(f"\n✅ {num_records} despesas extraídas\n")

if num_records == 0:
    print("⚠️ Sem dados")
    spark.createDataFrame([{
        "load_id": load_id, "entity_name": entity_name, "records_extracted": 0,
        "records_ingested": 0, "api_calls_count": 0, "checkpoint_before": None,
        "checkpoint_after": None, "execution_time_sec": int(time.time() - ingest_start),
        "status": "NO_NEW_DATA", "run_date": run_date, "env": env, "created_at": datetime.now()
    }]).write.mode("append").saveAsTable(f"{catalog}.{schema_log}.bronze_ingest_runs")
    dbutils.notebook.exit("NO_NEW_DATA")

print("🔧 Enriquecendo...")
enriched = []
ts = datetime.now()
dt = ts.date()

for item in data_list:
    pjson = json.dumps(item, ensure_ascii=False)
    phash = hashlib.sha256(pjson.encode('utf-8')).hexdigest()
    rid = str(item.get("id", "unknown"))
    dep_id = str(item.get("_deputado_id", "unknown"))
    
    enriched.append({
        "payload": pjson, "record_id": rid, "deputado_id": dep_id,
        "record_hash": phash, "load_id": load_id,
        "ingestion_timestamp": ts, "ingestion_date": dt,
        "source_endpoint": "/deputados/{id}/despesas", "env": env
    })

schema = StructType([
    StructField("payload", StringType()), StructField("record_id", StringType()),
    StructField("deputado_id", StringType()), StructField("record_hash", StringType()),
    StructField("load_id", StringType()), StructField("ingestion_timestamp", TimestampType()),
    StructField("ingestion_date", DateType()), StructField("source_endpoint", StringType()),
    StructField("env", StringType())
])

df_bronze = spark.createDataFrame(enriched, schema=schema)
print(f"✅ DataFrame: {df_bronze.count()} registros")
print("="*70)

## 💾 Bloco 8: Salvar

In [0]:
print(f"💾 Salvando...\n")
try:
    df_bronze.write.format("delta").mode("append").partitionBy("ingestion_date").saveAsTable(table_bronze)
    print(f"✅ Salvos: {num_records} registros")
    ingest_success = True
    ingest_error = None
except Exception as e:
    ingest_success = False
    ingest_error = str(e)
    print(f"❌ {e}")
    raise
finally:
    total_time = int(time.time() - ingest_start)
    print(f"⏱️ {total_time}s")
    print("="*70)

## 📝 Bloco 9: Logging

In [0]:
print("📝 Logging...\n")

api_calls = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog}.{schema_ops}.ingestion_requests
    WHERE load_id = '{load_id}' AND entity_name = '{entity_name}'
""").first()[0]

spark.createDataFrame([{
    "load_id": load_id, "entity_name": entity_name,
    "records_extracted": num_records, "records_ingested": num_records,
    "api_calls_count": api_calls,
    "checkpoint_before": f"{mes_processo:02d}/{ano_processo}",
    "checkpoint_after": f"{mes_processo:02d}/{ano_processo}",
    "execution_time_sec": total_time, "status": "SUCCESS",
    "error_message": ingest_error, "run_date": run_date, "env": env, "created_at": datetime.now()
}]).write.mode("append").saveAsTable(f"{catalog}.{schema_log}.bronze_ingest_runs")

print(f"✅ Log: {num_records} registros, {api_calls} calls")
print("="*70)

## ✅ Bloco 10: Validação

In [0]:
print("✅ Validando...\n")

total = spark.sql(f"SELECT COUNT(*) FROM {table_bronze}").first()[0]
this_load = spark.sql(f"SELECT COUNT(*) FROM {table_bronze} WHERE load_id = '{load_id}'").first()[0]

print(f"📊 Total: {total:,} | Este load: {this_load:,}")
print(f"   {'✅ OK' if this_load == num_records else '⚠️ DIVERGÊNCIA'}\n")

print("📊 Amostra:\n")
display(spark.sql(f"""
    SELECT record_id, deputado_id, LEFT(record_hash, 12) as hash, ingestion_date,
           LEFT(payload, 80) as preview
    FROM {table_bronze} WHERE load_id = '{load_id}' LIMIT 5
"""))

print(f"\n🎉 Notebook 05_bronze_ceap_despesas concluído!\n")
print(f"🎯 Período processado: {mes_processo:02d}/{ano_processo}")
print(f"🎯 Deputados processados: {len(deputados_ids)}")
print(f"🎯 Despesas ingeridas: {num_records:,}")